In [1]:
import pandas as pd

In [36]:
df = pd.read_csv('FakeNewsNet.csv',engine='python')

In [37]:
df.head()

,title,news_url,source_domain,tweet_num,real
0,Kandi Burruss Explodes Over Rape Accusation on...,http://toofab.com/2017/05/08/real-housewives-a...,toofab.com,42,1
1,People's Choice Awards 2018: The best red carp...,https://www.today.com/style/see-people-s-choic...,www.today.com,0,1
2,Sophia Bush Sends Sweet Birthday Message to 'O...,https://www.etonline.com/news/220806_sophia_bu...,www.etonline.com,63,1
3,Colombian singer Maluma sparks rumours of inap...,https://www.dailymail.co.uk/news/article-33655...,www.dailymail.co.uk,20,1
4,Gossip Girl 10 Years Later: How Upper East Sid...,https://www.zerchoo.com/entertainment/gossip-g...,www.zerchoo.com,38,1


In [5]:
df.isnull().sum()

,0
title,0
news_url,330
source_domain,330
tweet_num,0
real,0


In [ ]:
df.dropna()

In [7]:
X=df.drop('source_domain',axis=1)

In [38]:
y = df["real"].astype(int)

In [39]:
y

,real
0,1
1,1
2,1
3,1
4,1
...,...
23191,1
23192,0
23193,1
23194,0


In [10]:
import tensorflow as tf

In [11]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense

In [12]:
voc_size=5000

In [13]:
message = X.copy()

In [14]:
message['title'][1]

"People's Choice Awards 2018: The best red carpet looks"

In [15]:
import nltk
import re
from nltk.corpus import stopwords

In [16]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [17]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
corpus = []
for i in range(0,len(message)):
  review = re.sub('[^a-zA-Z]',' ',message['title'][i])
  review = review.lower()
  review = review.split()

  review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
  review = ' '.join(review)
  corpus.append(review)

In [18]:
onehot_rep = [one_hot(words,voc_size) for words in corpus]

In [ ]:
onehot_rep

In [20]:
sent=20
embed = pad_sequences(onehot_rep,padding='pre',maxlen=sent)

In [22]:
embed[0]

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0, 1157,
       3557, 4565, 2393, 3867, 3308, 3179, 3920,  318, 1742], dtype=int32)

In [23]:
embed_feature = 40
model = Sequential()
model.add(Embedding(voc_size,embed_feature,input_length=sent))
model.add(LSTM(100))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [24]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [25]:
len(embed)

23196

In [40]:
import numpy as np
X_final = np.array(embed)
y_final = np.array(y)

In [41]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train, y_test =train_test_split(X_final,y_final,test_size=0.33,random_state=42)

In [42]:
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=64)

Epoch 1/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 13s 41ms/step - accuracy: 0.7926 - loss: 0.4687 - val_accuracy: 0.8242 - val_loss: 0.4019
Epoch 2/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 10s 41ms/step - accuracy: 0.8494 - loss: 0.3486 - val_accuracy: 0.8265 - val_loss: 0.4031
Epoch 3/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 10s 40ms/step - accuracy: 0.8652 - loss: 0.3101 - val_accuracy: 0.8225 - val_loss: 0.4235
Epoch 4/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 8s 34ms/step - accuracy: 0.8758 - loss: 0.2827 - val_accuracy: 0.8244 - val_loss: 0.4389
Epoch 5/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 10s 42ms/step - accuracy: 0.8922 - loss: 0.2535 - val_accuracy: 0.8025 - val_loss: 0.4784
Epoch 6/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 10s 41ms/step - accuracy: 0.9021 - loss: 0.2269 - val_accuracy: 0.8107 - val_loss: 0.4754
Epoch 7/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 8s 35ms/step - accuracy: 0.9164 - loss: 0.2017 - val_accuracy: 0.7987 - val_loss: 0.5620
Epoch 8/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 11s 38ms/step - accuracy: 0.9249 - loss: 0.1789 - va

In [44]:
y_pred = model.predict(X_test)

240/240 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


In [45]:
y_pred = np.where(y_pred > 0.5, 1, 0)

In [46]:
from sklearn.metrics import confusion_matrix

In [47]:
confusion_matrix(y_test,y_pred)

array([[1043,  811],
       [ 775, 5026]])

In [49]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.7928151534944481

In [50]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.57      0.56      0.57      1854
           1       0.86      0.87      0.86      5801

    accuracy                           0.79      7655
   macro avg       0.72      0.71      0.72      7655
weighted avg       0.79      0.79      0.79      7655

